## CKY Parsing Step-by-Step

We will implement the CKY (Cocke-Kasami-Younger) parsing algorithm, which is a bottom-up parsing algorithm for context-free grammars in Chomsky Normal Form (CNF).

### Step 1: Define a Grammar in Chomsky Normal Form (CNF)

The CKY algorithm requires the grammar to be in Chomsky Normal Form. A grammar is in CNF if all production rules are of the form `A -> BC` (a non-terminal producing two non-terminals) or `A -> a` (a non-terminal producing a terminal symbol).

In [9]:
# Define a sample grammar in Chomsky Normal Form (CNF)
grammar = {
    'S': [['NP', 'VP']],
    'VP': [['V', 'NP'], ['V']],
    'NP': [['Det', 'N'], ['PropN']],
    'V': [['eats']],
    'N': [['cat'], ['dog']],
    'Det': [['the']],
    'PropN': [['Alice']]
}

# Example of how to access grammar rules
print("Rules for 'S':", grammar['S'])
print("Rules for 'V':", grammar['V'])

Rules for 'S': [['NP', 'VP']]
Rules for 'V': [['eats']]


### Step 2: Initialize the CKY Table and Fill the First 'Diagonal'

The CKY algorithm builds a table (often conceptualized as a triangle) to store all possible non-terminal derivations for each span of words. The first step is to fill the bottom-most row (or the main diagonal if you visualize it as a matrix) by checking which non-terminals can directly generate each word in the input sentence.

In [10]:
def cky_parse(sentence, grammar):
    words = sentence.split()
    n = len(words)

    # Initialize the CKY table: table[i][j] will store non-terminals covering words[i...j]
    # We'll use a list of lists of sets for convenience
    # table[length][start_index] where length goes from 1 to n
    # table[0] is for spans of length 1 (individual words)
    table = [[set() for _ in range(n)] for _ in range(n)]

    # Fill the first 'diagonal' (spans of length 1)
    for i, word in enumerate(words):
        for non_terminal, productions in grammar.items():
            for production in productions:
                if len(production) == 1 and production[0] == word:
                    table[0][i].add(non_terminal)

    # For demonstration, let's print the initialized table for the first row
    print("Initialized CKY table (first row - length 1 spans):")
    for i in range(n):
        print(f"Span '{words[i]}': {table[0][i]}")

    return table, words

# Test with a sample sentence
sentence = "the cat eats Alice"
cky_table, words = cky_parse(sentence, grammar)

Initialized CKY table (first row - length 1 spans):
Span 'the': {'Det'}
Span 'cat': {'N'}
Span 'eats': {'V'}
Span 'Alice': {'PropN'}


### Step 3: Fill the Remaining Cells of the CKY Table

Now, we'll iterate through all possible span lengths, from 2 up to the full length of the sentence. For each span, we'll consider all possible split points within that span. If we find non-terminals `B` and `C` in the left and right sub-spans respectively, and our grammar has a rule `A -> BC`, then `A` is added to the current cell (representing the combined span).

In [11]:
def cky_parse_full(sentence, grammar):
    words = sentence.split()
    n = len(words)

    # Initialize the CKY table: table[len-1][start_index] will store non-terminals
    # For a span of 'length' starting at 'start_index'
    table = [[set() for _ in range(n)] for _ in range(n)]

    # Fill the first 'diagonal' (spans of length 1)
    for i, word in enumerate(words):
        for non_terminal, productions in grammar.items():
            for production in productions:
                if len(production) == 1 and production[0] == word:
                    table[0][i].add(non_terminal)

    # Fill the rest of the table for spans of length 2 to n
    for length in range(2, n + 1):  # length goes from 2 up to n
        for i in range(n - length + 1):  # i is the start index of the span
            j = i + length - 1  # j is the end index of the span

            # k is the split point within the span [i...j]
            for k in range(i, j):
                # Non-terminals from the left sub-span (words[i...k])
                left_span_len = k - i + 1
                left_constituents = table[left_span_len - 1][i]

                # Non-terminals from the right sub-span (words[k+1...j])
                right_span_len = j - k
                right_constituents = table[right_span_len - 1][k + 1]

                # Check grammar rules of the form A -> BC
                for non_terminal_A, productions_A in grammar.items():
                    for production_rule in productions_A:
                        if len(production_rule) == 2:  # Check for A -> BC rules
                            B, C = production_rule[0], production_rule[1]
                            if B in left_constituents and C in right_constituents:
                                table[length - 1][i].add(non_terminal_A)

    # Print the full CKY table
    print("\nCKY Table:")
    for length_idx in range(n): # Iterate through lengths (0 to n-1 for table indexing)
        current_length = length_idx + 1
        print(f"Span Length {current_length}:")
        for start_idx in range(n - length_idx):
            end_idx = start_idx + current_length - 1
            span_words = ' '.join(words[start_idx : end_idx + 1])
            print(f"  [{start_idx},{end_idx}] '{span_words}': {table[length_idx][start_idx]}")

    # Check if the sentence is valid (S in the top-most cell)
    is_valid = 'S' in table[n - 1][0]
    print(f"\nSentence '{sentence}' is valid: {is_valid}")

    return table, is_valid

# Test with the sample sentence
sentence = "the cat eats Alice"
cky_table_final, is_sentence_valid = cky_parse_full(sentence, grammar)



CKY Table:
Span Length 1:
  [0,0] 'the': {'Det'}
  [1,1] 'cat': {'N'}
  [2,2] 'eats': {'V'}
  [3,3] 'Alice': {'PropN'}
Span Length 2:
  [0,1] 'the cat': {'NP'}
  [1,2] 'cat eats': set()
  [2,3] 'eats Alice': set()
Span Length 3:
  [0,2] 'the cat eats': set()
  [1,3] 'cat eats Alice': set()
Span Length 4:
  [0,3] 'the cat eats Alice': set()

Sentence 'the cat eats Alice' is valid: False


## Morphological CKY Parsing

Now, let's adapt the CKY parsing technique to analyze morphological features. This will involve reading a text file, segmenting words into morphemes, defining a morphological grammar, and then applying our CKY parser.

### Step 1: Text File Input and Basic Segmentation

We'll start by creating a dummy text file and a function to read its content. For morphological analysis, the input needs to be segmented into morphemes. A full morphological segmenter is complex; for this exercise, we'll begin with a very simplified segmenter that assumes words are already somewhat pre-processed or we'll define simple segmentation rules.

In [12]:
# Create a dummy text file for demonstration
file_content = "unhappily walked quickly"
with open('morph_input.txt', 'w') as f:
    f.write(file_content)

print("Created 'morph_input.txt' with content: ", file_content)

Created 'morph_input.txt' with content:  unhappily walked quickly


In [13]:
def read_text_file(filepath):
    """Reads content from a text file."""
    with open(filepath, 'r') as f:
        return f.read().strip()

def basic_morphological_segmenter(word):
    """
    A very basic placeholder for morphological segmentation.
    For now, it simply returns the word as its own morpheme,
    or applies very simple, hardcoded splits for demonstration.
    In a real system, this would be a sophisticated component.
    """
    # Example: A simplistic rule for known words
    if word == "unhappily":
        return ['un-', 'happy', '-ly']
    elif word == "walked":
        return ['walk', '-ed']
    elif word == "quickly":
        return ['quick', '-ly']
    else:
        return [word] # Default to treating the word as a single morpheme

# Read the text file
input_text = read_text_file('morph_input.txt')
print(f"\nInput text from file: '{input_text}'")

# Process each word with the basic segmenter
segmented_morphemes_list = []
for word in input_text.split():
    segmented_morphemes_list.extend(basic_morphological_segmenter(word))

print(f"Segmented morphemes: {segmented_morphemes_list}")

# We'll use this list of morphemes as our 'words' for the CKY parser in the next step


Input text from file: 'unhappily walked quickly'
Segmented morphemes: ['un-', 'happy', '-ly', 'walk', '-ed', 'quick', '-ly']


### Step 2: Define a Morphological Grammar in CNF

Now, we need a grammar that defines how morphemes combine according to Chomsky Normal Form (CNF). This grammar will specify how prefixes, roots, and suffixes form words. Our `basic_morphological_segmenter` provides us with the terminal morphemes, and this grammar will build up their higher-level structure.

In [14]:
# Define a morphological grammar in CNF
morph_grammar = {
    # Terminal productions (morphemes mapped to their categories)
    'PREFIX_NEG': [['un-']],
    'ROOT_ADJ': [['happy'], ['quick']],
    'ROOT_VERB': [['walk']],
    'SUFFIX_ADV': [['-ly']],
    'SUFFIX_PAST': [['-ed']],

    # Non-terminal productions (how categories combine in CNF)
    # These rules show how morpheme categories combine to form larger units

    # For adjectives that might be prefixed (e.g., 'unhappy')
    'ADJ_BASE': [['ROOT_ADJ']], # Represents a basic adjective root like 'happy' or 'quick'
    'ADJ_PREFIXED': [['PREFIX_NEG', 'ADJ_BASE']], # 'un-' + 'happy' -> 'unhappy'

    # For words formed with an adverbial suffix (e.g., 'happily', 'quickly', 'unhappily')
    'WORD_ADVERB': [
        ['ADJ_PREFIXED', 'SUFFIX_ADV'], # (unhappy) + (-ly) -> 'unhappily'
        ['ADJ_BASE', 'SUFFIX_ADV']      # (quick) + (-ly) -> 'quickly', (happy) + (-ly) -> 'happily'
    ],

    # For verbs in past tense (e.g., 'walked')
    'VERB_BASE': [['ROOT_VERB']], # Represents a basic verb root like 'walk'
    'WORD_PAST': [['VERB_BASE', 'SUFFIX_PAST']], # (walk) + (-ed) -> 'walked'

    # The top-level symbol for a complete morphologically parsed word
    'MORPH_WORD': [['WORD_ADVERB'], ['WORD_PAST']]
}

### Step 3: Adapt the CKY Parser for Morphological Analysis

We'll use a slightly adapted version of our `cky_parse_full` function. The core logic remains the same, but it will now operate on a list of morphemes (instead of words) and use our `morph_grammar`. The goal is to determine if the sequence of morphemes for a given word can be derived into a `MORPH_WORD` symbol.

In [15]:
def cky_morph_parse(morpheme_list, grammar, start_symbol='MORPH_WORD'):
    n = len(morpheme_list)
    if n == 0:
        return [], False

    # Initialize the CKY table: table[len-1][start_index] will store non-terminals
    # For a span of 'length' starting at 'start_index'
    table = [[set() for _ in range(n)] for _ in range(n)]

    # Helper function to apply unary rules until no more changes occur
    def apply_unary_rules(current_cell_set):
        changed = True
        while changed:
            changed = False
            new_additions = set()
            for current_nt in current_cell_set:
                for non_terminal_A, productions_A in grammar.items():
                    for production_rule in productions_A:
                        if len(production_rule) == 1 and production_rule[0] == current_nt:
                            if non_terminal_A not in current_cell_set and non_terminal_A not in new_additions:
                                new_additions.add(non_terminal_A)
                                changed = True
            current_cell_set.update(new_additions)
        return current_cell_set

    # Fill the first 'diagonal' (spans of length 1 - individual morphemes)
    for i, morpheme in enumerate(morpheme_list):
        for non_terminal, productions in grammar.items():
            for production in productions:
                if len(production) == 1 and production[0] == morpheme:
                    table[0][i].add(non_terminal)
        # Apply unary rules after initial terminal population for length 1 spans
        table[0][i] = apply_unary_rules(table[0][i])

    # Fill the rest of the table for spans of length 2 to n
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1

            for k in range(i, j):
                left_span_len = k - i + 1
                left_constituents = table[left_span_len - 1][i]

                right_span_len = j - k
                right_constituents = table[right_span_len - 1][k + 1]

                for non_terminal_A, productions_A in grammar.items():
                    for production_rule in productions_A:
                        if len(production_rule) == 2:
                            B, C = production_rule[0], production_rule[1]
                            if B in left_constituents and C in right_constituents:
                                table[length - 1][i].add(non_terminal_A)

            # Apply unary rules after binary rule population for all spans
            table[length - 1][i] = apply_unary_rules(table[length - 1][i])

    # Print the full CKY table for the current word's morphemes
    print(f"\nCKY Table for morphemes: {' '.join(morpheme_list)}")
    for length_idx in range(n): # Iterate through lengths (0 to n-1 for table indexing)
        current_length = length_idx + 1
        print(f"  Span Length {current_length}:")
        for start_idx in range(n - length_idx):
            end_idx = start_idx + current_length - 1
            span_morphemes = ' '.join(morpheme_list[start_idx : end_idx + 1])
            print(f"    [{start_idx},{end_idx}] '{span_morphemes}': {table[length_idx][start_idx]}")

    # Check if the morpheme sequence is valid (contains the start_symbol in the top-most cell)
    is_valid = start_symbol in table[n - 1][0]
    print(f"Morpheme sequence '{' '.join(morpheme_list)}' is morphologically valid: {is_valid}")

    return table, is_valid

### Step 4: Process the Text File and Analyze Morphological Features

Finally, we'll read the text file, segment each word, and apply our `cky_morph_parse` function to each word's morpheme sequence. This will demonstrate how the CKY algorithm can be used to analyze the internal (morphological) structure of words.

In [16]:
# Read the text file
input_text_for_morph = read_text_file('morph_input.txt')
print(f"\nProcessing input text: '{input_text_for_morph}'")

# Process each word individually
words_from_file = input_text_for_morph.split()

for word_to_parse in words_from_file:
    print(f"\n--- Analyzing word: '{word_to_parse}' ---")
    morphemes_for_word = basic_morphological_segmenter(word_to_parse)
    print(f"Segmented into morphemes: {morphemes_for_word}")

    if morphemes_for_word:
        morph_cky_table, morph_is_valid = cky_morph_parse(morphemes_for_word, morph_grammar)
    else:
        print(f"No morphemes found for '{word_to_parse}'.")

print("\n--- Morphological CKY Parsing Complete ---")


Processing input text: 'unhappily walked quickly'

--- Analyzing word: 'unhappily' ---
Segmented into morphemes: ['un-', 'happy', '-ly']

CKY Table for morphemes: un- happy -ly
  Span Length 1:
    [0,0] 'un-': {'PREFIX_NEG'}
    [1,1] 'happy': {'ROOT_ADJ', 'ADJ_BASE'}
    [2,2] '-ly': {'SUFFIX_ADV'}
  Span Length 2:
    [0,1] 'un- happy': {'ADJ_PREFIXED'}
    [1,2] 'happy -ly': {'WORD_ADVERB', 'MORPH_WORD'}
  Span Length 3:
    [0,2] 'un- happy -ly': {'WORD_ADVERB', 'MORPH_WORD'}
Morpheme sequence 'un- happy -ly' is morphologically valid: True

--- Analyzing word: 'walked' ---
Segmented into morphemes: ['walk', '-ed']

CKY Table for morphemes: walk -ed
  Span Length 1:
    [0,0] 'walk': {'VERB_BASE', 'ROOT_VERB'}
    [1,1] '-ed': {'SUFFIX_PAST'}
  Span Length 2:
    [0,1] 'walk -ed': {'MORPH_WORD', 'WORD_PAST'}
Morpheme sequence 'walk -ed' is morphologically valid: True

--- Analyzing word: 'quickly' ---
Segmented into morphemes: ['quick', '-ly']

CKY Table for morphemes: quick -ly
 